# Notebook 04: RevisitOP dataset feature cache

Prepare ROxford5k and RParis6k locally, then cache the frozen features required by the final selected model. Database images stay full-size; query images are cropped with the official boxes before preprocessing.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cbir.backbone import FrozenDinoV2Extractor
from cbir.cache import load_revisitop_feature_cache
from cbir.config import FusionConfig, config_to_dict, load_project_config
from cbir.data.revisitop import RevisitOPDataset
from cbir.data.revisitop_prepare import prepare_revisitop_datasets
from cbir.experiments import read_selection
from cbir.features import FeatureExtractionRunner
from cbir.plotting import SeriesData, plot_series
from cbir.utils import atomic_write_json
from cbir.workflow import build_revisitop_feature_cache, revisitop_feature_cache_location

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from the project root or notebooks directory.')
os.chdir(PROJECT_ROOT)
TORCH_HOME = PROJECT_ROOT / 'data' / 'models' / 'torch_hub'
os.environ.setdefault('TORCH_HOME', str(TORCH_HOME))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'local.yaml'
cfg = load_project_config(CONFIG_PATH)
FINAL_SELECTION_PATH = PROJECT_ROOT / 'outputs' / 'selections' / 'final_model_selection.json'
RAW_ARCHIVES_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'revisitop' / 'archives'
PREPARED_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'revisitop' / 'prepared'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '04_revisitop_feature_cache'
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_PATH = OUTPUT_DIR / 'results.json'
BACKBONE_BATCH_SIZE = 8
IMAGE_CHUNK_SIZE = 128
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

selection = read_selection(FINAL_SELECTION_PATH)
fusion_raw = dict(selection['fusion_config'])
fusion_raw['layer_indices'] = tuple(fusion_raw['layer_indices'])
fusion_config = FusionConfig(**fusion_raw)

print('Selected model:', selection['experiment_label'])
print('Selected layers (one-based):', [layer + 1 for layer in fusion_config.layer_indices])
print('Torch Hub cache:', os.environ['TORCH_HOME'])
print('Backbone batch size:', BACKBONE_BATCH_SIZE)

## Prepare the benchmark images

Place official archives under `data/raw/revisitop/archives/` to reuse them. Missing archives are downloaded there.

In [ ]:
preparation = prepare_revisitop_datasets(
    PREPARED_ROOT,
    archives_root=RAW_ARCHIVES_ROOT,
    datasets=('roxford5k', 'rparis6k'),
)
display(pd.DataFrame(preparation['datasets']).T)

## Extract or reuse frozen features

Raw images are decoded in small chunks. The resulting database and query tensors are loaded together during final evaluation.

In [ ]:
datasets = {}
locations = {}
for dataset_name in ('roxford5k', 'rparis6k'):
    dataset_root = PREPARED_ROOT / dataset_name
    dataset = RevisitOPDataset.from_ground_truth_pickle(
        name=dataset_name,
        ground_truth_path=dataset_root / f'gnd_{dataset_name}.pkl',
        image_root=dataset_root / 'jpg',
    )
    datasets[dataset_name] = dataset
    locations[dataset_name] = revisitop_feature_cache_location(
        cfg, dataset, layer_indices=fusion_config.layer_indices
    )

missing = []
bundles = {}
for dataset_name, location in locations.items():
    try:
        bundles[dataset_name] = load_revisitop_feature_cache(
            location.cache_dir, expected_fingerprint=location.fingerprint
        )
    except ValueError:
        missing.append(dataset_name)

if missing:
    runner = FeatureExtractionRunner(
        FrozenDinoV2Extractor(cfg.backbone), cfg.preprocess, cfg.pooling
    )
    for dataset_name in missing:
        dataset = datasets[dataset_name]
        location = build_revisitop_feature_cache(
            cfg,
            dataset,
            layer_indices=fusion_config.layer_indices,
            backbone_batch_size=BACKBONE_BATCH_SIZE,
            image_chunk_size=IMAGE_CHUNK_SIZE,
            runner=runner,
        )
        locations[dataset_name] = location
        bundles[dataset_name] = load_revisitop_feature_cache(
            location.cache_dir, expected_fingerprint=location.fingerprint
        )

cache_rows = [
    {
        'dataset': dataset_name,
        'status': 'extracted' if dataset_name in missing else 'reused',
        'database_images': len(bundles[dataset_name]['database']['image_ids']),
        'query_crops': len(bundles[dataset_name]['queries']['image_ids']),
        'cache_dir': str(locations[dataset_name].cache_dir),
        'fingerprint': locations[dataset_name].fingerprint,
    }
    for dataset_name in ('roxford5k', 'rparis6k')
]
atomic_write_json(RESULTS_PATH, {
    'selection': selection,
    'backbone_batch_size': BACKBONE_BATCH_SIZE,
    'image_chunk_size': IMAGE_CHUNK_SIZE,
    'caches': cache_rows,
    'config': config_to_dict(cfg),
})
display(pd.DataFrame(cache_rows))

In [ ]:
count_figure, count_axis = plot_series(
    {
        'Database images': SeriesData(
            x=[0, 1], y=[row['database_images'] for row in cache_rows]
        ),
        'Query crops': SeriesData(
            x=[0, 1], y=[row['query_crops'] for row in cache_rows]
        ),
    },
    title='Cached RevisitOP images',
    xlabel='Dataset',
    ylabel='Images',
    legend_title='Feature group',
    fig_size=(8, 5),
    save_path=FIGURES_DIR / 'cache_counts.png',
)
count_axis.set_xticks([0, 1], ['ROxford5k', 'RParis6k'])
display(count_figure)
plt.close(count_figure)